# [XX]: [Notebook Title]

**What this proves:** [One sentence summary of the key claim this notebook validates]

**Inputs:** [Configuration parameters, data sources, required artifacts from prior notebooks]

**Outputs:** [Artifacts produced: figures, metrics CSV, event logs, summary report]

**Runtime:** ~X min (demo) / ~Y min (paper)

**Goal:** [2-3 sentences describing what the reader will learn and why it matters]

## 1. Setup

In [ ]:
# Run mode: "demo" (quick sanity check) or "paper" (full statistical power)
RUN_MODE = "demo"  # @param ["demo", "paper"]

In [ ]:
import sys
from pathlib import Path

# Ensure notebook utilities are importable
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = NOTEBOOK_DIR.parent
sys.path.insert(0, str(NOTEBOOK_DIR))

# Notebook utilities
from _utils import (
    NotebookConfig,
    hash_config,
    print_config,
    print_repro_stamp,
    set_all_seeds,
    RunManager,
    paired_permutation_test,
    bootstrap_ci,
    effect_size,
    format_ci,
    setup_plotting,
    add_caption,
    COLORS,
    POLICY_COLORS,
)

# Reckon core imports
from reckon.core.resources import MemoryBudget
from reckon.memory import CausalMemoryManager, PayloadType
from reckon.memory import FIFOPolicy, LRUPolicy, UtilityPolicy, EvidencePerBytePolicy

# Plotting
import matplotlib.pyplot as plt
setup_plotting()

In [ ]:
# Configuration
config = NotebookConfig(
    notebook_name="XX_template",
    run_mode=RUN_MODE,
    seed=42,
    seed_count=10 if RUN_MODE == "paper" else 2,
    budget_slots=100,
    task_name="keydoor",
    num_episodes=100 if RUN_MODE == "paper" else 10,
    policy_name="evidence_per_byte",
)

print_config(config)
print_repro_stamp(config)
set_all_seeds(config.seed)

## 2. Problem + Hypotheses

[Describe the problem being investigated in 2-3 paragraphs. Explain why this matters and what gap in understanding this notebook addresses.]

**H1 ([Hypothesis Name]):** [Precise, testable statement about expected outcome]

**H2 ([Hypothesis Name]):** [Second hypothesis if applicable]

**Null Hypotheses:**
- H0₁: [What would it mean if H1 is false?]
- H0₂: [What would it mean if H2 is false?]

## 3. Minimal Theory

[Present the 2-3 key equations that underpin this notebook's experiments. Link to docs for full derivations.]

### 3.1 [First Concept]

[Brief explanation of the concept]

$$[\text{Key equation}]$$

**Code location:** `path/to/implementation.py` → `FunctionName`

### 3.2 [Second Concept]

$$[\text{Second key equation}]$$

For complete mathematical foundations, see [docs/03-Math-Core.md](../docs/03-Math-Core.md).

## 4. System Mapping

The following diagram shows how this notebook's experiments interact with the Reckon system.

```mermaid
flowchart TB
    subgraph ENV["Environment / Conversation"]
        E1["MiniGrid | TextWorld | Dialogue | Custom"]
    end
    
    subgraph ADAPT["Adapters"]
        A1["notebooks/_utils/adapters/"]
        A2["Convert env obs → (payload, payload_type, tags)"]
    end
    
    subgraph CMM["Causal Memory Manager<br/>src/reckon/memory/manager.py"]
        direction TB
        
        subgraph OPS["Operations"]
            W["write()"]
            R["read()"]
            P["prune()"]
            U["update_score()"]
        end
        
        POLICY["RetentionPolicy<br/>FIFO | LRU | Utility | Causal"]
        
        subgraph RUST["Rust Core"]
            STORE["RecordStore<br/>rust/src/store.rs<br/>O(1) lookup, O(log n) prune"]
            LOGGER["EventLogger<br/>rust/src/logging.rs<br/>sequence emission"]
        end
        
        OPS --> POLICY
        POLICY --> STORE
        STORE --> LOGGER
    end
    
    subgraph LOGS["JSONL Logs"]
        L1["runs/date/notebook/hash/logs/events.jsonl"]
    end
    
    subgraph UI["Debugger UI<br/>viz-ui/packages/debugger/"]
        U1["Timeline View"]
        U2["Causal Graph"]
        U3["Pareto Frontier"]
    end
    
    ENV --> ADAPT
    ADAPT --> CMM
    CMM --> LOGS
    LOGS --> UI
```

## 5. Experiment Design

**Unit of analysis:** [What constitutes one data point? e.g., "One episode completion with fixed seed"]

**Independent variables:**
- [Variable 1]: [Values/range]
- [Variable 2]: [Values/range]

**Dependent variables:**
- [Metric 1]: [Definition, range, direction of improvement]
- [Metric 2]: [Definition, range, direction of improvement]

**Seeds:** `{config.seed}` to `{config.seed + config.seed_count - 1}` ({config.seed_count} total)

**Statistical tests:**
- Paired permutation test (α = 0.05, n_permutations = 10000)
- Bootstrap 95% CI (n_bootstrap = 10000)
- Effect size: Cohen's d

**Power analysis:** [If applicable, explain sample size justification]

## 6. Demo Run

A quick sanity check that runs in both demo and paper modes. This verifies the experimental setup before committing to full runs.

In [ ]:
# Quick sanity check (always runs)
print("Demo Run: Verifying experimental setup")
print("=" * 50)

budget = MemoryBudget(max_slots=config.budget_slots)
cmm = CausalMemoryManager(budget, episode_id="demo-1")

# Write some test records
for i in range(5):
    cmm.set_step(i)
    cmm.write(
        payload={"test": i},
        payload_type=PayloadType.OBSERVATION,
        tags=["demo"],
        initial_utility=0.1 * i,
    )

print(f"Records stored: {cmm.current_usage_slots}")
print(f"Bytes used: {cmm.current_usage_bytes}")
print(f"Events logged: {len(cmm.get_event_log())}")
print("\nDemo run complete. Setup verified.")

## 7. Full Run (Paper Mode)

The complete experiment with full statistical power. Only runs when `RUN_MODE = "paper"`.

In [ ]:
if RUN_MODE == "paper":
    print("Running full experiment...")
    print(f"Seeds: {config.seed} to {config.seed + config.seed_count - 1}")
    print(f"Episodes per seed: {config.num_episodes}")
    print()
    
    # [Full experiment code here]
    # results = []
    # for seed in get_seed_range(config.seed, config.seed_count):
    #     set_all_seeds(seed)
    #     result = run_experiment(...)
    #     results.append(result)
    
    print("Full run complete.")
else:
    print("Skipping full run (demo mode)")
    print("Set RUN_MODE = 'paper' and restart kernel for complete results.")

## 8. Results

[Present results with statistical rigor. Include effect sizes, confidence intervals, and p-values.]

In [ ]:
# Example results visualization
fig, ax = plt.subplots(figsize=(8, 5))

# Example data
budgets = [10, 20, 50, 100]
performance = [0.45, 0.62, 0.78, 0.85]

ax.plot(budgets, performance, 'o-', color=COLORS['primary'], linewidth=2, markersize=8)
ax.set_xlabel("Memory Budget (slots)")
ax.set_ylabel("Success Rate")
ax.set_title("Performance vs Budget")

add_caption(
    ax,
    "[Description of what this plot shows]",
    n=len(budgets),
    seeds=f"{config.seed}-{config.seed + config.seed_count - 1}",
    test="permutation",
    mode=RUN_MODE,
    policy=config.policy_name,
    budget=config.budget_slots,
)

plt.tight_layout()
plt.show()

In [ ]:
# Statistical summary
print("Statistical Summary")
print("=" * 50)
print()
print("H1 ([Hypothesis Name]):")
print("  Result: [CONFIRMED/REJECTED]")
print("  p-value: [value]")
print("  Effect size (Cohen's d): [value]")
print("  95% CI: [lower, upper]")
print()
print("H2 ([Hypothesis Name]):")
print("  Result: [CONFIRMED/REJECTED]")
print("  ...")

## 9. Debugger Walkthrough

The event logs produced by this notebook can be loaded into the Reckon debugger for interactive exploration.

### Starting the Debugger

```bash
# Terminal 1: Start the debugger UI
cd viz-ui && pnpm dev

# Terminal 2: Serve the logs directory
python -m http.server 8080 --directory notebooks/runs/

# Open http://localhost:5173 and load the JSONL file from the artifacts directory
```

### What to Look For

When viewing the logs from this notebook:

1. **[Specific feature to examine]:** [What it reveals about the experiment]
2. **[Another feature]:** [Its significance]
3. **[Third feature]:** [Why it matters]

## 10. Limitations + Next Steps

### Limitations

1. **[Limitation 1]:** [Explanation of what this limitation means for interpreting results]

2. **[Limitation 2]:** [Explanation]

3. **[Limitation 3]:** [Explanation]

### Next Steps

- [ ] **[Next experiment]:** [What it would test and why]
- [ ] **[Another direction]:** [What it would reveal]
- [ ] **[Third direction]:** [Why it matters]

## Export Artifacts

In [ ]:
# Save all artifacts
run = RunManager(config)
run.save_config()

# Save metrics
metrics = {
    "demo_records": cmm.current_usage_slots,
    "run_mode": RUN_MODE,
}
run.save_metrics(metrics)

# Save event logs
events = cmm.get_event_log()
if events:
    run.save_logs(events)

# Save figure (if created)
# run.save_figure(fig, "results_plot")

# Write summary
run.write_summary(
    key_metrics=metrics,
    notes="[Brief description of what this run demonstrated]"
)

run.print_artifacts()

## Further Reading

- [docs/03-Math-Core.md](../docs/03-Math-Core.md) — Mathematical foundations
- [docs/11-Evaluation-Metrics.md](../docs/11-Evaluation-Metrics.md) — Causal Record Contract
- [docs/12-Reproducibility.md](../docs/12-Reproducibility.md) — Canonical ordering
- [ARCHITECTURE.md](../ARCHITECTURE.md) — System design overview